<!-- Step1: Introduction.
In this Project, we are solving a binary classification problem: predicting whether a potential client (a lead) will sign up for the platform (converted = 1) or not (converted = 0). -->

In [1]:
%mamba install pandas numpy matplotlib seaborn

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas, seaborn
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 12.627 seconds
  Name          Version  Build                Channel
-------------------------------------------------------------------
+ pandas        3.1.0    np23py313h124dbc7_0  emscripten-forge-4x
+ patsy         1.0.3    py313h1804a44_0      emscripten-forge-4x
+ seaborn       0.13.2   hd8ed1ab_3           conda-forge
+ seaborn-base  0.13.2   pyhd8ed1ab_3         conda-forge
+ six           1.17.0   pyhe01879c_1         conda-forge
+ statsmodels   0.15.0   np23py313h07ed413_0  emscripten-forge-4x
- pip           26.2.2   pyh145f28c_0         conda-forge


In [5]:
# Step2: Data Preparation.
 # 2.1 Code: Setup & Downloading the Data
import pandas as pd
import numpy as np

df = pd.read_csv("course_lead_scoring_2026.csv")
print(f"Dataset shape: {df.shape}")
df.head()

# 3.2: Data Preparation & Cleaning.
# 1. Standardize Column Names & Values
df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

# 2. Identify Categorical vs. Numerical Columns
# Automatically identify categorical and numerical columns
categorical = df.select_dtypes(include=['object', 'string', 'category']).columns
numerical = df.select_dtypes(include=['int64', 'float64']).columns

# Display the columns
print("Categorical Columns:")
print(list(categorical))

print("\nNumerical Columns:")
print(list(numerical))

# Differentiate categorical and numerical columns
for col in df.columns:
    if col in categorical:
        print(f'{col} (Categorical): {df[col].isnull().sum()}')
    elif col in numerical:
        print(f'{col} (Numerical): {df[col].isnull().sum()}')

# 3. Handle Missing Values (Homework Requirement)
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

# Verify no missing values remain
print("No Missing Values Remain: ")
print(df.isnull().sum())

print(df.dtypes)
print("\nUnique values in target 'converted':", df['converted'].unique())


Dataset shape: (5000, 9)
Categorical Columns:
['lead_source', 'industry', 'employment_status', 'location']

Numerical Columns:
['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score', 'converted']
lead_source (Categorical): 148
industry (Categorical): 240
employment_status (Categorical): 194
location (Categorical): 208
annual_income (Numerical): 369
number_of_courses_viewed (Numerical): 0
interaction_count (Numerical): 0
lead_score (Numerical): 35
converted (Numerical): 0
No Missing Values Remain: 
lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64
lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of

In [6]:
!mamba install scikit-learn

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas, seaborn, scikit-learn
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 6.526 seconds
  Name                Version     Build                Channel
----------------------------------------------------------------------------
+ ca-certificates     2026.11.20  hbd8a1cb_0           conda-forge
+ certifi             2026.11.20  pyhd8ed1ab_0         conda-forge
+ charset-normalizer  3.5.1       pyhd8ed1ab_0         conda-forge
+ idna                3.20        pyh5ded981_0         conda-forge
+ joblib              1.6.0       py313h57e9779_0      emscripten-forge-4x
+ narwhals            2.29.0      pyh5ded981_0         conda-forge
+ requests            2.34.2      pyhcf101f3_0         conda-forge
+ scikit-learn        1.9.1       np23py313hde19dca_0  emscripten-forge-4x
+ setuptools          84.2.1      pyhe730653_0         conda-forge
+ threadpoolctl 

In [7]:
# Step 3: Setting Up the Validation Framework
from sklearn.model_selection import train_test_split

# 1. Split into 80% full_train and 20% test
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

# 2. Split full_train into 60% train and 20% validation (0.25 of 0.80 = 0.20)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

# 3. Verify split sizes
print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

# 4. Reset indices so row numbers stay clean
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# 5. Extract the target variable ('converted') as NumPy arrays
y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

# 6. Remove the target variable from the feature DataFrames to prevent data leakage
del df_train['converted']
del df_val['converted']
del df_test['converted']


Train: 3000, Val: 1000, Test: 1000


In [8]:
 # Step 4: (Exploratory Data Analysis — EDA)
# 1. Check Target Variable Distribution (Conversion Rate)
# First, let's check the overall proportion of leads that converted (converted == 1) vs. 
# didn't convert (converted == 0):
df_full_train = df_full_train.reset_index(drop=True)

# Check class distribution
print(df_full_train['converted'].value_counts(normalize=True))

# Global conversion rate (mean of 0s and 1s)
global_conversion_rate = df_full_train['converted'].mean()
print(f"\nGlobal conversion rate: {global_conversion_rate:.4f}")


# 2. Inspect Unique Categories & Answer Homework Q1
# Let's check how many unique values each categorical feature has, and find the most frequent observation (mode) 
# for the industry column:
# Number of unique values per categorical column
print(df_full_train[categorical].nunique())

# Frequency counts for 'industry' (note: check on df or df_full_train as specified)
print("\nIndustry value counts (full dataset):")
print(df['industry'].value_counts())

print("\nIndustry mode:", df['industry'].mode()[0])


converted
1    0.57075
0    0.42925
Name: proportion, dtype: float64

Global conversion rate: 0.5707
lead_source          6
industry             7
employment_status    5
location             6
dtype: int64

Industry value counts (full dataset):
industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

Industry mode: technology


# Step 5 (Feature Importance: Conversion Rate & Risk Ratio)
<!-- In this step, we look at how the target rate (converted) varies across each category within our categorical columns (lead_source, industry, employment_status, location) compared to the global conversion rate.

Understanding the Two Metrics

• Difference (group_mean - global_mean):
	• Positive (> 0): Leads in this group are more likely to convert than average.
	• Negative (< 0): Leads in this group are less likely to convert than average.
• Risk / Conversion Ratio (group_mean / global_mean):
	• Above 1.0: Higher likelihood of converting than average (e.g., 1.25 means 25% more likely).
	• Below 1.0: Lower likelihood of converting than average (e.g., 0.75 means 25% less likely). -->

In [9]:
from IPython.display import display

global_conversion = df_full_train['converted'].mean()

for c in categorical:
    print(f" {c.upper()} ")
    df_group = df_full_train.groupby(c)['converted'].agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - global_conversion
    df_group['risk_ratio'] = df_group['mean'] / global_conversion
    display(df_group.sort_values(by='risk_ratio', ascending=False))
    print()


 LEAD_SOURCE 


,mean,count,diff,risk_ratio
lead_source,,,,
referral,0.763844,614,0.193094,1.338316
organic_search,0.677579,1008,0.106829,1.187174
NA,0.610169,118,0.039419,1.069066
social_media,0.547962,834,-0.022788,0.960073
paid_ads,0.427957,930,-0.142793,0.749815
events,0.411290,496,-0.159460,0.720614



 INDUSTRY 


,mean,count,diff,risk_ratio
industry,,,,
finance,0.613402,582,0.042652,1.074730
technology,0.611111,918,0.040361,1.070716
NA,0.587629,194,0.016879,1.029573
healthcare,0.559398,665,-0.011352,0.980111
retail,0.553741,735,-0.017009,0.970200
manufacturing,0.548969,388,-0.021781,0.961838
education,0.500000,518,-0.070750,0.876040



 EMPLOYMENT_STATUS 


,mean,count,diff,risk_ratio
employment_status,,,,
employed,0.656459,2090,0.085709,1.150170
NA,0.580247,162,0.009497,1.016639
self_employed,0.548276,580,-0.022474,0.960623
unemployed,0.435572,551,-0.135178,0.763157
student,0.419773,617,-0.150977,0.735476



 LOCATION 


,mean,count,diff,risk_ratio
location,,,,
NA,0.603550,169,0.032800,1.057469
asia,0.585308,844,0.014558,1.025507
europe,0.584821,896,0.014071,1.024654
north_america,0.576923,1092,0.006173,1.010816
africa,0.546154,390,-0.024596,0.956906
south_america,0.525452,609,-0.045298,0.920633


<!-- • lead_source (Strongest categorical signal): Conversion rates span a wide ~35 percentage point range, from 76.4% (referral, risk_ratio = 1.34) down to 41.1% (events, risk_ratio = 0.72).
• employment_status (Strong signal): Huge gap between employed (65.6%, risk_ratio = 1.15) and student (42.0%, risk_ratio = 0.74).
• industry (Moderate signal): Narrower spread, ranging from 61.3% (finance) to 50.0% (education).
• location (Weakest signal): Almost flat across regions (risk_ratio stays tightly clustered between 0.92 and 1.06). --> -->


In [10]:
# Feature Importance : Mutual Information & Correlation
# 1. Mutual Information (Categorical Features)
from sklearn.metrics import mutual_info_score

def mutual_info_converted_score(series):
    return mutual_info_score(series, df_full_train['converted'])

# Note: check whether your homework prompt asks to calculate MI on df_full_train or df_train
mi_full = df_full_train[categorical].apply(mutual_info_converted_score).sort_values(ascending=False)
print(" Mutual Information (df_full_train) ")
print(mi_full.round(2))

mi_train = df_train[categorical].apply(lambda s: mutual_info_score(s, y_train)).sort_values(ascending=False)
print("\n Mutual Information (df_train) ")
print(mi_train.round(2))


# 2. Correlation (Numerical Features)
# Correlation of numerical features with the target 'converted'
print(" Correlation with target 'converted' ")
print(df_full_train[numerical].corrwith(df_full_train['converted']).sort_values(ascending=False))

# Correlation matrix among numerical features (Homework Q2)
print("\n Numerical Feature Correlation Matrix ")
display(df_full_train[numerical].corr())


 Mutual Information (df_full_train) 
lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

 Mutual Information (df_train) 
lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64
 Correlation with target 'converted' 
converted                   1.000000
lead_score                  0.482078
interaction_count           0.450273
number_of_courses_viewed    0.393200
annual_income               0.188437
dtype: float64

 Numerical Feature Correlation Matrix 


,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.163680,0.122528,0.222493,0.188437
number_of_courses_viewed,0.163680,1.000000,0.723040,0.757178,0.393200
interaction_count,0.122528,0.723040,1.000000,0.916626,0.450273
lead_score,0.222493,0.757178,0.916626,1.000000,0.482078
converted,0.188437,0.393200,0.450273,0.482078,1.000000


In [11]:
# Step 7: One-Hot Encoding with DictVectorizer
from sklearn.feature_extraction import DictVectorizer

# Ensure 'numerical' only includes the 4 feature columns (not 'converted')
numerical = [
    'number_of_courses_viewed',
    'annual_income',
    'interaction_count',
    'lead_score'
]

features = ['lead_source', 'industry', 'employment_status', 'location',
            'number_of_courses_viewed', 'annual_income', 'interaction_count', 'lead_score']

# 1. Convert DataFrames to lists of dictionaries
train_dicts = df_train[features].to_dict(orient='records')
val_dicts = df_val[features].to_dict(orient='records')

# 2. Fit DictVectorizer on training data ONLY, then transform both train and val
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

# 3. Inspect the resulting feature matrix shapes and names
print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print("\nEncoded feature names:")
print(dv.get_feature_names_out())


X_train shape: (3000, 28)
X_val shape: (1000, 28)

Encoded feature names:
['annual_income' 'employment_status=NA' 'employment_status=employed'
 'employment_status=self_employed' 'employment_status=student'
 'employment_status=unemployed' 'industry=NA' 'industry=education'
 'industry=finance' 'industry=healthcare' 'industry=manufacturing'
 'industry=retail' 'industry=technology' 'interaction_count' 'lead_score'
 'lead_source=NA' 'lead_source=events' 'lead_source=organic_search'
 'lead_source=paid_ads' 'lead_source=referral' 'lead_source=social_media'
 'location=NA' 'location=africa' 'location=asia' 'location=europe'
 'location=north_america' 'location=south_america'
 'number_of_courses_viewed']


In [13]:
# Step 8 (Logistic Regression)
def sigmoid(z):
    return 1 / (1 + np.exp(-z))



In [15]:
# Step 9: Training Logistic Regression with Scikit-Learn.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Initialize and train the model using the homework hyperparameters
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# 2. Predict probabilities on the validation set
# predict_proba returns two columns: [P(converted=0), P(converted=1)]
y_pred_proba = model.predict_proba(X_val)[:, 1]

# 3. Convert probabilities to binary predictions using a 0.5 threshold
y_pred = (y_pred_proba >= 0.5)

# 4. Calculate validation accuracy
val_accuracy = accuracy_score(y_val, y_pred)
# (Equivalent to: (y_val == y_pred).mean())

print(f"Validation Accuracy: {val_accuracy:.4f}")


Validation Accuracy: 0.6450


In [16]:
# Quick Sanity Check on Predictions
df_val_results = pd.DataFrame({
    'probability': y_pred_proba.round(3),
    'prediction': y_pred.astype(int),
    'actual': y_val
})
df_val_results['correct'] = df_val_results['prediction'] == df_val_results['actual']
df_val_results.head(10)


,probability,prediction,actual,correct
0,0.617,1,1,True
1,0.444,0,1,False
2,0.604,1,0,False
3,0.646,1,1,True
4,0.522,1,0,False
5,0.779,1,1,True
6,0.607,1,1,True
7,0.545,1,0,False
8,0.768,1,1,True
9,0.528,1,1,True


In [18]:
 # Step 10: (Model Interpretation: Weights & Bias)
# In this step, we inspect what the model actually learned (intercept_ and coef_) and run 
# leave-one-out feature elimination to see which features help or hurt accuracy 
# (directly answering Homework Question 5).

# 1. Inspect Bias (w₀) and Feature Weights (w)
# Pair dv.get_feature_names_out() with model.coef_[0] to see how each feature pushes the conversion probability up or down:
# Bias term (w_0)
print(f"Bias (intercept): {model.intercept_[0]:.4f}\n")

# Pair feature names with their learned weights
weights = dict(zip(dv.get_feature_names_out(), model.coef_[0].round(4)))
df_weights = pd.DataFrame(list(weights.items()), columns=['feature', 'weight'])
display(df_weights.sort_values(by='weight', ascending=False))


# 2. Feature Elimination (Homework Question 5)
# Next, train the exact same model while excluding one feature at a time from features, 
# and record the difference from your original accuracy (0.6450):

orig_acc = val_accuracy
all_features = list(categorical) + list(numerical)

results = []

for f in all_features:
    subset = [col for col in all_features if col != f]
    
    dv_sub = DictVectorizer(sparse=False)
    X_train_sub = dv_sub.fit_transform(df_train[subset].to_dict(orient='records'))
    X_val_sub = dv_sub.transform(df_val[subset].to_dict(orient='records'))
    
    m = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    m.fit(X_train_sub, y_train)
    
    acc = accuracy_score(y_val, m.predict(X_val_sub))
    diff = orig_acc - acc
    
    results.append({
        'removed_feature': f,
        'accuracy': round(acc, 4),
        'diff (orig - new)': round(diff, 4),
        'abs_diff': round(abs(diff), 4)
    })

df_elim = pd.DataFrame(results).sort_values(by='abs_diff', ascending=True)
display(df_elim)


Bias (intercept): -0.0108



,feature,weight
13,interaction_count,0.1367
27,number_of_courses_viewed,0.0538
14,lead_score,0.0110
19,lead_source=referral,0.0087
2,employment_status=employed,0.0074
17,lead_source=organic_search,0.0069
21,location=NA,0.0002
15,lead_source=NA,0.0001
0,annual_income,-0.0000
6,industry=NA,-0.0004


,removed_feature,accuracy,diff (orig - new),abs_diff
1,industry,0.645,0.000,0.000
2,employment_status,0.645,0.000,0.000
3,location,0.645,0.000,0.000
7,lead_score,0.644,0.001,0.001
4,number_of_courses_viewed,0.643,0.002,0.002
0,lead_source,0.642,0.003,0.003
6,interaction_count,0.601,0.044,0.044
5,annual_income,0.724,-0.079,0.079


<!-- • Removing industry, employment_status, or location produces 0.000 change in validation accuracy (meaning they have the smallest difference—a direct answer for Homework Question 5!).
• Removing annual_income actually jumps accuracy from 0.645 to 0.724 (diff = -0.079) because its huge unscaled values (~50,000+) dominate the L2 regularization penalty in liblinear and squeeze the other feature weights down near zero! -->

<!-- • Removing industry, employment_status, or location produces 0.000 change in validation accuracy (meaning they have the smallest difference—a direct answer for Homework Question 5!).
• Removing annual_income actually jumps accuracy from 0.645 to 0.724 (diff = -0.079) because its huge unscaled values (~50,000+) dominate the L2 regularization penalty in liblinear and squeeze the other feature weights down near zero! -->

In [19]:
# Step 11: Regularization Tuning (C) & Training the Final Model
# 1. Tune Parameter C (Homework Question 6)
# Remember: smaller C = stronger regularization; larger C = weaker regularization:

C_values = [0.01, 0.1, 1, 10, 100]
c_results = []

for c in C_values:
    m = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    m.fit(X_train, y_train)
    
    acc = accuracy_score(y_val, m.predict(X_val))
    c_results.append({
        'C': c,
        'val_accuracy': round(acc, 3),
        'exact_accuracy': round(acc, 6)
    })

df_c = pd.DataFrame(c_results)
display(df_c)


# 2. Train the Final Model on df_full_train

best_C = df_c.sort_values(by=['val_accuracy', 'C'], ascending=[False, True]).iloc[0]['C']
print(f"Selected best C: {best_C}")

# Prepare full_train and test sets
y_full_train = df_full_train['converted'].values
del df_full_train['converted']

dv_final = DictVectorizer(sparse=False)
X_full_train = dv_final.fit_transform(df_full_train[all_features].to_dict(orient='records'))
X_test = dv_final.transform(df_test[all_features].to_dict(orient='records'))

final_model = LogisticRegression(solver='liblinear', C=best_C, max_iter=1000, random_state=42)
final_model.fit(X_full_train, y_full_train)

test_accuracy = accuracy_score(y_test, final_model.predict(X_test))
print(f"Final Test Accuracy: {test_accuracy:.4f}")


,C,val_accuracy,exact_accuracy
0,0.01,0.645,0.645
1,0.10,0.645,0.645
2,1.00,0.645,0.645
3,10.00,0.645,0.645
4,100.00,0.645,0.645


Selected best C: 0.01
Final Test Accuracy: 0.6700
